# Thesis Experiments — Chapter 4 per-dataset reruns

Reruns the Chapter-4 experiment plan from `thesis_model (1).ipynb` Sections 5–6 **per-dataset** — no merged pool. Every architecture is trained twice: once on Derm7pt, once on MILK10k.

| Chapter 4 | Variant | Class | Checkpoint stem |
|---|---|---|---|
| 4.1.1 | Single-branch RGB | `SingleBranchRGBClassifier` | `exp_<ds>_singlergb` |
| 4.1.2 | Dual concat baseline | `DualBranchBaseline` | `exp_<ds>_baseline` |
| 4.2 | SE-ResNet + concat | `DualBranchSEResNet` | `exp_<ds>_channel` |
| 4.3.1 | Additive fusion | `DualBranchElementwiseFusion('add')` | `exp_<ds>_addfusion` |
| 4.3.2 | Multiplicative fusion | `DualBranchElementwiseFusion('mul')` | `exp_<ds>_mulfusion` |
| 4.3.3 | BiCrossAttn | `DualBranchBiCrossAttn` | `exp_<ds>_bicross` |
| 4.4 | SE + CrossAttn combined | `DualBranchSECrossCombined` | `exp_<ds>_combined` |

`<ds>` is `derm7pt` or `milk10k`. 4.4 ablation table is derived from variants 2/3/6/7 — no extra training.

## 1 — Setup, data prep, transforms

In [ ]:
import os, copy, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, balanced_accuracy_score,
    cohen_kappa_score, matthews_corrcoef, log_loss
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available(): torch.backends.cudnn.benchmark = True
print(f'Device: {device}')

DERM7PT_DIR = Path('dataset') / 'Derm7pt'
MILK_DIR    = Path('dataset') / 'milk10k'

# Derm7pt
derm_df = pd.read_csv(DERM7PT_DIR / 'meta' / 'meta.csv')
derm_df['clinic_path'] = derm_df['clinic'].apply(lambda x: str(DERM7PT_DIR / 'images' / x))
derm_df['derm_path']   = derm_df['derm'].apply(lambda x: str(DERM7PT_DIR / 'images' / x))
derm_df['diagnosis']   = derm_df['diagnosis'].str.strip().str.lower()
diagnosis_groups = {
    'melanoma': 'MEL', 'melanoma (less than 0.76 mm)': 'MEL', 'melanoma (in situ)': 'MEL',
    'melanoma (0.76 to 1.5 mm)': 'MEL', 'melanoma (more than 1.5 mm)': 'MEL',
    'melanoma metastasis': 'MEL',
    'clark nevus': 'NV', 'reed or spitz nevus': 'NV', 'dermal nevus': 'NV',
    'blue nevus': 'NV', 'congenital nevus': 'NV', 'combined nevus': 'NV',
    'recurrent nevus': 'NV',
    'basal cell carcinoma': 'BCC',
    'seborrheic keratosis': 'SK',
    'lentigo': 'MISC', 'dermatofibroma': 'MISC', 'vascular lesion': 'MISC',
    'melanosis': 'MISC', 'miscellaneous': 'MISC',
}
derm_df['diagnosis_group'] = derm_df['diagnosis'].map(diagnosis_groups)
class_names = sorted(derm_df['diagnosis_group'].dropna().unique())
label_map   = {name: i for i, name in enumerate(class_names)}
NC = len(class_names)

# MILK10k
milk_meta = pd.read_csv(MILK_DIR / 'MILK10k_Training_Metadata.csv')
milk_gt   = pd.read_csv(MILK_DIR / 'MILK10k_Training_GroundTruth.csv')
milk_class_map = {'MEL':'MEL','NV':'NV','BCC':'BCC','BKL':'SK','DF':'MISC','VASC':'MISC'}
drop_classes   = {'AKIEC','SCCKA','INF','BEN_OTH','MAL_OTH'}
gt_cols = [c for c in milk_gt.columns if c != 'lesion_id']
milk_gt['raw_class'] = milk_gt[gt_cols].idxmax(axis=1)
milk_gt = milk_gt[~milk_gt['raw_class'].isin(drop_classes)].copy()
milk_gt['diagnosis_group'] = milk_gt['raw_class'].map(milk_class_map)
clinic_meta = (milk_meta[milk_meta['image_type']=='clinical: close-up']
               [['lesion_id','isic_id']].rename(columns={'isic_id':'clinic_isic'}))
derm_meta   = (milk_meta[milk_meta['image_type']=='dermoscopic']
               [['lesion_id','isic_id']].rename(columns={'isic_id':'derm_isic'}))
paths = clinic_meta.merge(derm_meta, on='lesion_id')
milk_df = milk_gt[['lesion_id','diagnosis_group']].merge(paths, on='lesion_id')
milk_df['clinic_path'] = milk_df.apply(
    lambda r: str(MILK_DIR / 'MILK10k_Training_Input' / r['lesion_id'] / f"{r['clinic_isic']}.jpg"), axis=1)
milk_df['derm_path'] = milk_df.apply(
    lambda r: str(MILK_DIR / 'MILK10k_Training_Input' / r['lesion_id'] / f"{r['derm_isic']}.jpg"), axis=1)
milk_df['source'] = 'MILK10k'

derm7_clean = derm_df[['clinic_path','derm_path','diagnosis_group']].copy()
derm7_clean['source'] = 'Derm7pt'
combined = pd.concat(
    [derm7_clean, milk_df[['clinic_path','derm_path','diagnosis_group','source']]],
    ignore_index=True)
combined['label'] = combined['diagnosis_group'].map(label_map)
df = combined.dropna(subset=['label']).reset_index(drop=True)
print(f'Pool: {len(df)} samples  ('
      f"Derm7pt={int((df['source']=='Derm7pt').sum())}, "
      f"MILK10k={int((df['source']=='MILK10k').sum())})  | {NC} classes")

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])


class SkinLesionDualDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clinic_img = Image.open(row['clinic_path']).convert('RGB')
        derm_img   = Image.open(row['derm_path']).convert('RGB')
        if self.transform:
            clinic_img = self.transform(clinic_img)
            derm_img   = self.transform(derm_img)
        return clinic_img, derm_img, torch.tensor(row['label'], dtype=torch.long)


## 2 — Hyperparameters + per-dataset loader factory

In [ ]:
LR             = 0.000750
DROPOUT        = 0.32
EMBED_DIM      = 256
NUM_HEADS      = 4
WEIGHT_DECAY   = 0.000957
BATCH_SIZE     = 64
FINAL_EPOCHS   = 60
PATIENCE       = 20
UNFREEZE_EPOCH = 5
GRAD_CLIP      = 1.0

_NUM_WORKERS = 0 if device.type in ('mps','cpu') or os.name=='nt' else min(8, os.cpu_count() or 4)
_PIN_MEMORY  = device.type == 'cuda'


def split_indices(sub):
    # Stratified 70/15/15 split, SEED=42 — same scheme everywhere.
    y = sub['label'].values.astype(int)
    idx = np.arange(len(sub))
    tr_i, tmp_i = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
    va_i, te_i  = train_test_split(tmp_i, test_size=0.50, stratify=y[tmp_i], random_state=SEED)
    return tr_i, va_i, te_i


def make_dataset_loaders(source, batch_size=BATCH_SIZE):
    # Train/val/test loaders + loss weights for ONE dataset (Derm7pt or MILK10k).
    assert source in ('Derm7pt', 'MILK10k'), 'per-dataset only — no merged'
    sub = df[df['source'] == source].reset_index(drop=True)
    tr_i, va_i, te_i = split_indices(sub)
    tr_y = sub['label'].values.astype(int)[tr_i]
    counts = np.bincount(tr_y, minlength=NC)
    samp_w = (1.0 / np.where(counts == 0, 1, counts))[tr_y]
    loss_w = torch.tensor(
        [len(tr_y) / (NC * c) if c > 0 else 0.0 for c in counts],
        dtype=torch.float32).to(device)
    tr_ds = SkinLesionDualDataset(sub.iloc[tr_i], train_transform)
    va_ds = SkinLesionDualDataset(sub.iloc[va_i], val_transform)
    te_ds = SkinLesionDualDataset(sub.iloc[te_i], val_transform)
    sampler = WeightedRandomSampler(samp_w, len(tr_y), replacement=True)
    kw = dict(num_workers=_NUM_WORKERS, pin_memory=_PIN_MEMORY,
              persistent_workers=(_NUM_WORKERS > 0))
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, sampler=sampler, **kw)
    va_ld = DataLoader(va_ds, batch_size=batch_size, shuffle=False, **kw)
    te_ld = DataLoader(te_ds, batch_size=batch_size, shuffle=False, **kw)
    print(f'  {source}: {len(sub)} samples -> '
          f'train {len(tr_i)} / val {len(va_i)} / test {len(te_i)}')
    return tr_ld, va_ld, te_ld, loss_w


print(f'Setup ready. FINAL_EPOCHS={FINAL_EPOCHS}')


## 3 — Model definitions

All 7 Chapter-4 architectures defined once. Lifted from `thesis_model (1).ipynb` Sections 5 & 6 verbatim.

In [ ]:
class ResNet50Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4)
    def forward(self, x): return self.features(x)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.sigmoid(self.fc(self.avg_pool(x).view(b, c))).view(b, c, 1, 1)
        return x * w


class SEBottleneck(nn.Module):
    def __init__(self, bottleneck, reduction=16):
        super().__init__()
        self.block = bottleneck
        self.se    = SEBlock(bottleneck.conv3.out_channels, reduction)
    def forward(self, x):
        identity = x
        out = self.block.conv1(x); out = self.block.bn1(out); out = self.block.relu(out)
        out = self.block.conv2(out); out = self.block.bn2(out); out = self.block.relu(out)
        out = self.block.conv3(out); out = self.block.bn3(out)
        out = self.se(out)
        if self.block.downsample is not None:
            identity = self.block.downsample(x)
        out = out + identity
        return self.block.relu(out)


class SEResNet50Backbone(nn.Module):
    def __init__(self, pretrained=True, reduction=16):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        for layer in [base.layer1, base.layer2, base.layer3, base.layer4]:
            for i in range(len(layer)):
                layer[i] = SEBottleneck(layer[i], reduction=reduction)
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4)
    def forward(self, x): return self.features(x)


class CrossAttentionFusion(nn.Module):
    def __init__(self, in_dim=2048, embed_dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.q_proj = nn.Linear(in_dim, embed_dim)
        self.k_proj = nn.Linear(in_dim, embed_dim)
        self.v_proj = nn.Linear(in_dim, embed_dim)
        self.attn   = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1  = nn.LayerNorm(embed_dim)
        self.ffn    = nn.Sequential(nn.Linear(embed_dim, embed_dim*2), nn.GELU(),
                                     nn.Dropout(dropout), nn.Linear(embed_dim*2, embed_dim))
        self.norm2 = nn.LayerNorm(embed_dim)
        self.pool  = nn.AdaptiveAvgPool1d(1)
    def forward(self, clinic_feat, derm_feat):
        c = clinic_feat.flatten(2).transpose(1, 2)
        d = derm_feat.flatten(2).transpose(1, 2)
        Q = self.q_proj(c); K = self.k_proj(d); V = self.v_proj(d)
        attn_out, attn_weights = self.attn(Q, K, V, need_weights=True, average_attn_weights=True)
        x = self.norm1(Q + attn_out)
        x = self.norm2(x + self.ffn(x))
        fused = self.pool(x.transpose(1, 2)).squeeze(-1)
        return fused, attn_weights


class BidirectionalCrossAttnFusion(nn.Module):
    def __init__(self, in_dim=2048, embed_dim=256, num_heads=4, dropout=0.32):
        super().__init__()
        self.c2d  = CrossAttentionFusion(in_dim, embed_dim, num_heads, dropout)
        self.d2c  = CrossAttentionFusion(in_dim, embed_dim, num_heads, dropout)
        self.proj = nn.Linear(embed_dim*2, embed_dim)
    def forward(self, clinic_feat, derm_feat):
        fused_c2d, attn_c2d = self.c2d(clinic_feat, derm_feat)
        fused_d2c, attn_d2c = self.d2c(derm_feat, clinic_feat)
        fused = self.proj(torch.cat([fused_c2d, fused_d2c], dim=1))
        return fused, (attn_c2d, attn_d2c)


# 4.1.1 — Single-branch RGB
class SingleBranchRGBClassifier(nn.Module):
    def __init__(self, num_classes=5, dropout=0.32, pretrained=True):
        super().__init__()
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(512, num_classes))
    def _freeze_backbones(self):
        for p in self.resnet_clinic.parameters(): p.requires_grad = False
        for p in self.resnet_clinic.features[7].parameters(): p.requires_grad = True
    def unfreeze_resnets(self):
        for p in self.resnet_clinic.features[6].parameters(): p.requires_grad = True
    def forward(self, clinic_img, derm_img=None):
        x = self.pool(self.resnet_clinic(clinic_img)).flatten(1)
        return self.classifier(x), None


# 4.1.2 — Dual concat baseline
class DualBranchBaseline(nn.Module):
    def __init__(self, num_classes=5, dropout=0.32, pretrained=True):
        super().__init__()
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = ResNet50Backbone(pretrained=pretrained)
        self.fusion = nn.Sequential(
            nn.Conv2d(4096, 1024, kernel_size=1, bias=False),
            nn.BatchNorm2d(1024), nn.ReLU(inplace=True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(512, num_classes))
    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters(): p.requires_grad = False
            for p in m.features[7].parameters(): p.requires_grad = True
    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters(): p.requires_grad = True
    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        x = self.fusion(torch.cat([feat_c, feat_d], dim=1))
        x = self.pool(x).flatten(1)
        return self.classifier(x), None


# 4.2 — SE-ResNet + concat
class DualBranchSEResNet(nn.Module):
    def __init__(self, num_classes=5, dropout=0.32, pretrained=True):
        super().__init__()
        self.resnet_clinic = SEResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = SEResNet50Backbone(pretrained=pretrained)
        self.fusion = nn.Sequential(
            nn.Conv2d(4096, 1024, kernel_size=1, bias=False),
            nn.BatchNorm2d(1024), nn.ReLU(inplace=True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(512, num_classes))
    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters(): p.requires_grad = False
            for p in m.features[7].parameters(): p.requires_grad = True
            for module in m.modules():
                if isinstance(module, SEBlock):
                    for p in module.parameters(): p.requires_grad = True
    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters(): p.requires_grad = True
    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        x = self.fusion(torch.cat([feat_c, feat_d], dim=1))
        x = self.pool(x).flatten(1)
        return self.classifier(x), None


# 4.3.1 / 4.3.2 — Elementwise fusion
class DualBranchElementwiseFusion(nn.Module):
    def __init__(self, num_classes=5, dropout=0.32, fusion='add', pretrained=True):
        super().__init__()
        assert fusion in ('add', 'mul')
        self.fusion = fusion
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = ResNet50Backbone(pretrained=pretrained)
        self.post_fuse = nn.Sequential(nn.BatchNorm2d(2048), nn.ReLU(inplace=True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(512, num_classes))
    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters(): p.requires_grad = False
            for p in m.features[7].parameters(): p.requires_grad = True
    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters(): p.requires_grad = True
    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        fused = feat_c + feat_d if self.fusion == 'add' else feat_c * feat_d
        x = self.pool(self.post_fuse(fused)).flatten(1)
        return self.classifier(x), None


# 4.3.3 — Bidirectional Cross-Attention
class DualBranchBiCrossAttn(nn.Module):
    def __init__(self, num_classes=5, embed_dim=256, num_heads=4, dropout=0.32, pretrained=True):
        super().__init__()
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = ResNet50Backbone(pretrained=pretrained)
        self.fusion        = BidirectionalCrossAttnFusion(2048, embed_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.BatchNorm1d(embed_dim // 2),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes))
    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters(): p.requires_grad = False
            for p in m.features[7].parameters(): p.requires_grad = True
    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters(): p.requires_grad = True
    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        fused, attn_weights = self.fusion(feat_c, feat_d)
        return self.classifier(fused), attn_weights


# 4.4 — Full: SE + BiCrossAttn
class DualBranchSECrossCombined(nn.Module):
    def __init__(self, num_classes=5, embed_dim=256, num_heads=4, dropout=0.32, pretrained=True):
        super().__init__()
        self.resnet_clinic = SEResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = SEResNet50Backbone(pretrained=pretrained)
        self.fusion        = BidirectionalCrossAttnFusion(2048, embed_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.BatchNorm1d(embed_dim // 2),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes))
    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters(): p.requires_grad = False
            for p in m.features[7].parameters(): p.requires_grad = True
            for module in m.modules():
                if isinstance(module, SEBlock):
                    for p in module.parameters(): p.requires_grad = True
    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters(): p.requires_grad = True
    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        fused, attn_weights = self.fusion(feat_c, feat_d)
        return self.classifier(fused), attn_weights


print('7 architectures defined.')


## 4 — Training utilities + ARCHS registry

In [ ]:
class EarlyStopping:
    def __init__(self, patience=20, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.counter = 0; self.best_loss = float('inf')
        self.best_state = None; self.stop = False
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True
    def restore(self, model):
        if self.best_state is not None: model.load_state_dict(self.best_state)


def train_one_epoch(model, loader, optimizer, criterion, grad_clip):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for clinic_imgs, derm_imgs, lb in loader:
        clinic_imgs = clinic_imgs.to(device); derm_imgs = derm_imgs.to(device); lb = lb.to(device)
        optimizer.zero_grad()
        logits, _ = model(clinic_imgs, derm_imgs)
        loss = criterion(logits, lb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item() * clinic_imgs.size(0)
        correct += logits.argmax(1).eq(lb).sum().item()
        total += clinic_imgs.size(0)
    return total_loss / total, correct / total


def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for clinic_imgs, derm_imgs, lb in loader:
            clinic_imgs = clinic_imgs.to(device); derm_imgs = derm_imgs.to(device); lb = lb.to(device)
            logits, _ = model(clinic_imgs, derm_imgs)
            loss = criterion(logits, lb)
            total_loss += loss.item() * clinic_imgs.size(0)
            correct += logits.argmax(1).eq(lb).sum().item()
            total += clinic_imgs.size(0)
    return total_loss / total, correct / total


def run_training(model, train_loader, val_loader, loss_weights, ckpt_path, tag):
    # Warm-up freeze -> unfreeze layer3 at UNFREEZE_EPOCH -> cosine LR + early stop.
    model = model.to(device)
    model._freeze_backbones()
    criterion = nn.CrossEntropyLoss(weight=loss_weights)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                           lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCH)
    early_stop = EarlyStopping(patience=PATIENCE)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    early_stop_epoch = None
    start = time.time()

    for epoch in range(FINAL_EPOCHS):
        if epoch == UNFREEZE_EPOCH:
            model.unfreeze_resnets()
            optimizer = optim.Adam(model.parameters(), lr=LR / 4, weight_decay=WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=FINAL_EPOCHS - UNFREEZE_EPOCH)

        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, GRAD_CLIP)
        va_loss, va_acc = validate(model, val_loader, criterion)
        scheduler.step()
        history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)

        early_stop(va_loss, model)
        if epoch == 0 or (epoch + 1) % 5 == 0 or early_stop.stop:
            print(f'    ep {epoch+1:2d}/{FINAL_EPOCHS} | '
                  f'train {tr_loss:.3f}/{tr_acc:.3f} | val {va_loss:.3f}/{va_acc:.3f} | '
                  f'es {early_stop.counter}/{PATIENCE}')
        if early_stop.stop:
            early_stop_epoch = epoch + 1
            early_stop.restore(model)
            break

    print(f'  {tag}: {(time.time()-start)/60:.1f} min | '
          f'best val loss {early_stop.best_loss:.4f} | '
          f'epochs {early_stop_epoch or FINAL_EPOCHS}')
    torch.save({
        'model_state_dict': model.state_dict(),
        'class_names':      class_names,
        'label_map':        label_map,
        'history':          history,
        'best_val_loss':    early_stop.best_loss,
        'early_stop_epoch': early_stop_epoch,
    }, ckpt_path)
    print(f'  saved -> {ckpt_path}')
    return model


ARCHS = [
    ('singlergb',  SingleBranchRGBClassifier,   dict(num_classes=NC, dropout=DROPOUT)),
    ('baseline',   DualBranchBaseline,          dict(num_classes=NC, dropout=DROPOUT)),
    ('channel',    DualBranchSEResNet,          dict(num_classes=NC, dropout=DROPOUT)),
    ('addfusion',  DualBranchElementwiseFusion, dict(num_classes=NC, dropout=DROPOUT, fusion='add')),
    ('mulfusion',  DualBranchElementwiseFusion, dict(num_classes=NC, dropout=DROPOUT, fusion='mul')),
    ('bicross',    DualBranchBiCrossAttn,       dict(num_classes=NC, embed_dim=EMBED_DIM,
                                                     num_heads=NUM_HEADS, dropout=DROPOUT)),
    ('combined',   DualBranchSECrossCombined,   dict(num_classes=NC, embed_dim=EMBED_DIM,
                                                     num_heads=NUM_HEADS, dropout=DROPOUT)),
]
print('Ready to train:', [k for k, _, _ in ARCHS])


## 5 — Train all 7 variants on **Derm7pt** (Dataset 1)

Derm7pt runs first (smaller, ~1k cases → quick smoke-test feedback). Checkpoints: `exp_derm7pt_<key>.pth`.

In [ ]:
print('=== Dataset 1: Derm7pt ===')
d_tr, d_va, d_te, d_lw = make_dataset_loaders('Derm7pt', BATCH_SIZE)

for key, cls, kwargs in ARCHS:
    ckpt = f'exp_derm7pt_{key}.pth'
    print(f'\n--- Derm7pt / {key} ---')
    run_training(cls(**kwargs), d_tr, d_va, d_lw, ckpt, f'Derm7pt/{key}')
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nDerm7pt complete — 7 checkpoints saved.')


## 6 — Train all 7 variants on **MILK10k** (Dataset 2)

Same loop on MILK10k. Checkpoints: `exp_milk10k_<key>.pth`.

In [ ]:
print('=== Dataset 2: MILK10k ===')
m_tr, m_va, m_te, m_lw = make_dataset_loaders('MILK10k', BATCH_SIZE)

for key, cls, kwargs in ARCHS:
    ckpt = f'exp_milk10k_{key}.pth'
    print(f'\n--- MILK10k / {key} ---')
    run_training(cls(**kwargs), m_tr, m_va, m_lw, ckpt, f'MILK10k/{key}')
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nMILK10k complete — 7 checkpoints saved.')


## 7 — Evaluation: all 7 × 2 datasets

Rebuilds the test split for each dataset with the same SEED, loads each `exp_<ds>_<key>.pth`, falls back to `ablation_<ds>_<key>.pth` if present. Missing checkpoints are skipped — table still renders for whatever exists. Outputs `experiments_metrics.csv` and `chapter4_experiments.png`.

In [ ]:
ARCH_LABELS = [
    ('singlergb', 'Single-RGB (4.1.1)'),
    ('baseline',  'Baseline (4.1.2)'),
    ('channel',   'SE-ResNet (4.2)'),
    ('addfusion', 'Add-Fusion (4.3.1)'),
    ('mulfusion', 'Mul-Fusion (4.3.2)'),
    ('bicross',   'BiCrossAttn (4.3.3)'),
    ('combined',  'Combined (4.4)'),
]
ARCH_CLS = {
    'singlergb': SingleBranchRGBClassifier,
    'baseline':  DualBranchBaseline,
    'channel':   DualBranchSEResNet,
    'addfusion': DualBranchElementwiseFusion,
    'mulfusion': DualBranchElementwiseFusion,
    'bicross':   DualBranchBiCrossAttn,
    'combined':  DualBranchSECrossCombined,
}
ARCH_KW = {
    'singlergb': dict(num_classes=NC, dropout=DROPOUT, pretrained=False),
    'baseline':  dict(num_classes=NC, dropout=DROPOUT, pretrained=False),
    'channel':   dict(num_classes=NC, dropout=DROPOUT, pretrained=False),
    'addfusion': dict(num_classes=NC, dropout=DROPOUT, fusion='add', pretrained=False),
    'mulfusion': dict(num_classes=NC, dropout=DROPOUT, fusion='mul', pretrained=False),
    'bicross':   dict(num_classes=NC, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                      dropout=DROPOUT, pretrained=False),
    'combined':  dict(num_classes=NC, embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                      dropout=DROPOUT, pretrained=False),
}


def build_test_loader(source, batch_size=64):
    sub = df[df['source'] == source].reset_index(drop=True)
    _, _, te_i = split_indices(sub)
    te_ds = SkinLesionDualDataset(sub.iloc[te_i], val_transform)
    return DataLoader(te_ds, batch_size=batch_size, shuffle=False,
                      num_workers=_NUM_WORKERS, pin_memory=(device.type == 'cuda'))


def find_ckpt(dset_key, arch_key):
    for path in (f'exp_{dset_key}_{arch_key}.pth',
                 f'ablation_{dset_key}_{arch_key}.pth'):
        if Path(path).exists():
            return path
    return None


def evaluate_ckpt(model, ckpt_path, loader):
    model = model.to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    preds, y, probs = [], [], []
    with torch.no_grad():
        for ci, di, lb in loader:
            logits, _ = model(ci.to(device), di.to(device))
            p = F.softmax(logits, dim=1)
            preds.extend(logits.argmax(1).cpu().numpy())
            y.extend(lb.numpy()); probs.extend(p.cpu().numpy())
    y = np.array(y); preds = np.array(preds); probs = np.array(probs)
    return {
        'accuracy':          accuracy_score(y, preds),
        'balanced_accuracy': balanced_accuracy_score(y, preds),
        'f1_macro':          f1_score(y, preds, average='macro',    zero_division=0),
        'f1_weighted':       f1_score(y, preds, average='weighted', zero_division=0),
        'cohen_kappa':       cohen_kappa_score(y, preds),
        'mcc':               matthews_corrcoef(y, preds),
        'log_loss':          log_loss(y, probs, labels=list(range(NC))),
    }


DATASETS = [('Derm7pt', 'derm7pt'), ('MILK10k', 'milk10k')]
test_loaders = {dlabel: build_test_loader(dlabel) for dlabel, _ in DATASETS}
for dlabel, _ in DATASETS:
    print(f'{dlabel} test samples: {len(test_loaders[dlabel].dataset)}')

rows = []
for akey, alabel in ARCH_LABELS:
    for dlabel, dkey in DATASETS:
        ckpt = find_ckpt(dkey, akey)
        if ckpt is None:
            print(f'  {alabel:20s} / {dlabel}: no checkpoint - skipping')
            continue
        m = evaluate_ckpt(ARCH_CLS[akey](**ARCH_KW[akey]), ckpt, test_loaders[dlabel])
        m['Architecture'] = alabel; m['Dataset'] = dlabel; m['ckpt'] = ckpt
        rows.append(m)
        print(f'  {alabel:20s} / {dlabel:8s} | acc={m["accuracy"]:.4f}  '
              f'f1_macro={m["f1_macro"]:.4f}  kappa={m["cohen_kappa"]:.4f}  ({ckpt})')
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if rows:
    metric_cols = ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted',
                   'cohen_kappa', 'mcc', 'log_loss']
    res = pd.DataFrame(rows)
    res['Row'] = res['Architecture'] + ' / ' + res['Dataset']
    tbl = res.set_index('Row')[metric_cols]
    tbl.to_csv('experiments_metrics.csv')

    fig = plt.figure(figsize=(15, max(11, len(tbl) * 0.55 + 6)))
    gs = fig.add_gridspec(2, 1, height_ratios=[len(tbl) + 2, 7], hspace=0.30)
    fig.suptitle('Chapter 4 — All 7 variants x 2 datasets (per-dataset held-out test)',
                 fontsize=15, fontweight='bold', y=0.99)

    ax_t = fig.add_subplot(gs[0]); ax_t.axis('off')
    col_labels = ['Accuracy', 'Balanced Acc', 'F1 (macro)', 'F1 (weighted)',
                  'Cohen kappa', 'MCC', 'Log loss']
    cell_text = [[f'{v:.4f}' for v in row] for row in tbl.values]
    table = ax_t.table(cellText=cell_text, rowLabels=tbl.index, colLabels=col_labels,
                       cellLoc='center', rowLoc='center', loc='center')
    table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1, 1.5)
    for j in range(len(col_labels)):
        table[0, j].set_facecolor('#34495e')
        table[0, j].set_text_props(color='white', fontweight='bold')
    for dlabel, _ in DATASETS:
        sub_tbl = tbl[tbl.index.str.endswith(f'/ {dlabel}')]
        if sub_tbl.empty: continue
        for j, col in enumerate(metric_cols):
            best_i = sub_tbl[col].idxmin() if col == 'log_loss' else sub_tbl[col].idxmax()
            r = list(tbl.index).index(best_i)
            table[r + 1, j].set_facecolor('#d5f5e3')
            table[r + 1, j].set_text_props(fontweight='bold')
    ax_t.set_title('Green = best per metric per dataset   |   Log loss: lower is better',
                   fontsize=9, pad=8)

    ax_b = fig.add_subplot(gs[1])
    pivot = res.pivot(index='Architecture', columns='Dataset', values='f1_macro')
    pivot = pivot.reindex(index=[lbl for _, lbl in ARCH_LABELS],
                          columns=[d for d, _ in DATASETS if d in pivot.columns])
    pivot.plot(kind='bar', ax=ax_b, rot=20, colormap='Set2', width=0.75)
    ax_b.set_ylabel('F1 (macro)'); ax_b.set_ylim(0, 1)
    ax_b.set_xlabel('')
    ax_b.set_title('F1 (macro) by architecture x dataset', fontsize=11)
    ax_b.legend(title='Dataset', loc='lower right')
    ax_b.grid(alpha=0.3, axis='y')
    for cont in ax_b.containers:
        ax_b.bar_label(cont, fmt='%.3f', fontsize=8, padding=2)

    plt.savefig('chapter4_experiments.png', dpi=200, bbox_inches='tight')
    print('\nSaved -> experiments_metrics.csv')
    print('Saved -> chapter4_experiments.png')
    plt.show()
else:
    print('No checkpoints found — run cells 5 and 6 first.')


## 8 — Section 4.4 ablation view (derived)

Four configurations × two datasets — populated from checkpoints already evaluated above. No retraining.

| Row | Channel attn (SE) | Cross attn | Source variant |
|---|---|---|---|
| Full model | yes | yes | 4.4 combined |
| Without Cross-Attn | yes | no | 4.2 SE-ResNet |
| Without SE | no | yes | 4.3.3 BiCrossAttn |
| Plain concat | no | no | 4.1.2 baseline |

Outputs `chapter4_ablation.csv` and `chapter4_ablation.png`.

In [ ]:
ABLATION = [
    ('Full (SE + CrossAttn)', 'combined',  'yes', 'yes'),
    ('Without Cross-Attn',    'channel',   'yes', 'no'),
    ('Without SE',            'bicross',   'no',  'yes'),
    ('Plain concat',          'baseline',  'no',  'no'),
]

ablation_rows = []
for label, akey, se, ca in ABLATION:
    for dlabel, dkey in DATASETS:
        ckpt = find_ckpt(dkey, akey)
        if ckpt is None:
            print(f'  {label:24s} / {dlabel}: no checkpoint - skipping')
            continue
        m = evaluate_ckpt(ARCH_CLS[akey](**ARCH_KW[akey]), ckpt, test_loaders[dlabel])
        m['Ablation'] = label; m['SE'] = se; m['Cross-Attn'] = ca; m['Dataset'] = dlabel
        ablation_rows.append(m)
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if ablation_rows:
    metric_cols = ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted',
                   'cohen_kappa', 'mcc', 'log_loss']
    ares = pd.DataFrame(ablation_rows)
    ares['Row'] = ares['Ablation'] + ' / ' + ares['Dataset']
    atbl = ares.set_index('Row')[['SE', 'Cross-Attn'] + metric_cols]
    atbl.to_csv('chapter4_ablation.csv')

    fig = plt.figure(figsize=(15, max(8, len(atbl) * 0.55 + 5)))
    gs = fig.add_gridspec(2, 1, height_ratios=[len(atbl) + 2, 6], hspace=0.30)
    fig.suptitle('Section 4.4 — Ablation: SE-ResNet x Cross-Attention x Dataset',
                 fontsize=15, fontweight='bold', y=0.99)

    ax_t = fig.add_subplot(gs[0]); ax_t.axis('off')
    col_labels = ['SE', 'Cross-Attn', 'Accuracy', 'Balanced Acc', 'F1 (macro)',
                  'F1 (weighted)', 'Cohen kappa', 'MCC', 'Log loss']
    cell_text = []
    for row in atbl.values:
        cell_text.append([str(row[0]), str(row[1])] + [f'{v:.4f}' for v in row[2:]])
    table = ax_t.table(cellText=cell_text, rowLabels=atbl.index, colLabels=col_labels,
                       cellLoc='center', rowLoc='center', loc='center')
    table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1, 1.5)
    for j in range(len(col_labels)):
        table[0, j].set_facecolor('#34495e')
        table[0, j].set_text_props(color='white', fontweight='bold')
    for dlabel, _ in DATASETS:
        sub_tbl = atbl[atbl.index.str.endswith(f'/ {dlabel}')]
        if sub_tbl.empty: continue
        for j, col in enumerate(metric_cols):
            best_i = sub_tbl[col].idxmin() if col == 'log_loss' else sub_tbl[col].idxmax()
            r = list(atbl.index).index(best_i)
            table[r + 1, j + 2].set_facecolor('#d5f5e3')
            table[r + 1, j + 2].set_text_props(fontweight='bold')
    ax_t.set_title('Green = best per metric per dataset   |   Log loss: lower is better',
                   fontsize=9, pad=8)

    ax_b = fig.add_subplot(gs[1])
    pivot = ares.pivot(index='Ablation', columns='Dataset', values='f1_macro')
    pivot = pivot.reindex(index=[lbl for lbl, *_ in ABLATION],
                          columns=[d for d, _ in DATASETS if d in pivot.columns])
    pivot.plot(kind='bar', ax=ax_b, rot=15, colormap='Set2', width=0.75)
    ax_b.set_ylabel('F1 (macro)'); ax_b.set_ylim(0, 1)
    ax_b.set_xlabel('')
    ax_b.set_title('F1 (macro) by ablation x dataset', fontsize=11)
    ax_b.legend(title='Dataset', loc='lower right')
    ax_b.grid(alpha=0.3, axis='y')
    for cont in ax_b.containers:
        ax_b.bar_label(cont, fmt='%.3f', fontsize=8, padding=2)

    plt.savefig('chapter4_ablation.png', dpi=200, bbox_inches='tight')
    print('\nSaved -> chapter4_ablation.csv')
    print('Saved -> chapter4_ablation.png')
    plt.show()
else:
    print('No ablation checkpoints found — run cells 5 and 6 first.')
